In [1]:
#!/usr/bin/env python
# ═══════════════════════════════════════════════════════════════════════
#  IMAGE RESTORATION — INFERENCE-ONLY SUBMISSION NOTEBOOK
# ═══════════════════════════════════════════════════════════════════════
#  This notebook loads pretrained weights and generates the submission.
#  No training — runs in ~5 minutes with TTA, ~1 min without.
#
#  SETUP: Upload your model_weights.pth as a Kaggle Dataset and add it
#         as an input to this notebook. Then update WEIGHTS_PATH below.
# ═══════════════════════════════════════════════════════════════════════

# Image Restoration — Submission Notebook
Loads pretrained model → runs inference on test set → creates submission.csv

In [2]:
import os, glob, math, time, random
import numpy as np
import pandas as pd
import base64
from io import BytesIO

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ═══════════════════════════════════════════════════════════════════════
#  ★ UPDATE THIS PATH to match your uploaded dataset name ★
# ═══════════════════════════════════════════════════════════════════════
WEIGHTS_PATH = "/kaggle/input/datasets/khxjamohammed/nafnet-mini/best_model (2).pth"
# If your dataset is named differently, adjust accordingly, e.g.:
# WEIGHTS_PATH = "/kaggle/input/my-sr-weights/model_weights.pth"

# ── Competition paths ──
DATA_ROOT = "/kaggle/input/competitions/ExeBit_kla_ai_hack/KLA AI - HACK"
TEST_LR   = os.path.join(DATA_ROOT, "test", "NoisyLR")
SUB_DIR   = "/kaggle/working/submission"
os.makedirs(SUB_DIR, exist_ok=True)

USE_TTA = True   # 8× self-ensemble (set False for faster but lower quality)

test_files = sorted(os.listdir(TEST_LR))
print(f"Test images: {len(test_files)}")
print(f"Weights:     {WEIGHTS_PATH}")
print(f"TTA:         {'ON (8×)' if USE_TTA else 'OFF'}")

Device: cuda
Test images: 200
Weights:     /kaggle/input/datasets/khxjamohammed/nafnet-mini/best_model (2).pth
TTA:         ON (8×)


In [3]:

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class SimplifiedChannelAttention(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        return x * self.fc(self.pool(x))


class NAFBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.norm = nn.GroupNorm(1, ch)
        self.body = nn.Sequential(
            nn.Conv2d(ch, ch * 2, 3, padding=1),
            SimpleGate(),
            nn.Conv2d(ch, ch, 3, padding=1),
        )
        self.sca   = SimplifiedChannelAttention(ch)
        self.scale = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        res = self.body(self.norm(x))
        res = self.sca(res)
        return x + res * self.scale


class AdvancedUNetSR(nn.Module):
    """
    U-Net with NAFNet-style blocks + PixelShuffle ×2.
    Input:  (B, 1, 128, 128) → Output: (B, 1, 256, 256)
    """
    def __init__(self, ch=64, n_blocks=6):
        super().__init__()

        # Encoder
        self.head = nn.Conv2d(1, ch, 3, padding=1)
        self.enc1 = nn.Sequential(*[NAFBlock(ch) for _ in range(n_blocks)])
        self.down1 = nn.Conv2d(ch, ch*2, 2, stride=2)

        self.enc2 = nn.Sequential(*[NAFBlock(ch*2) for _ in range(n_blocks)])
        self.down2 = nn.Conv2d(ch*2, ch*4, 2, stride=2)

        # Bottleneck
        self.mid = nn.Sequential(*[NAFBlock(ch*4) for _ in range(n_blocks)])

        # Decoder
        self.up2 = nn.Sequential(
            nn.Conv2d(ch*4, ch*2 * 4, 1),
            nn.PixelShuffle(2),
        )
        self.fuse2 = nn.Conv2d(ch*4, ch*2, 1)
        self.dec2 = nn.Sequential(*[NAFBlock(ch*2) for _ in range(n_blocks)])

        self.up1 = nn.Sequential(
            nn.Conv2d(ch*2, ch * 4, 1),
            nn.PixelShuffle(2),
        )
        self.fuse1 = nn.Conv2d(ch*2, ch, 1)
        self.dec1 = nn.Sequential(*[NAFBlock(ch) for _ in range(n_blocks)])

        # 2× SR upsample
        self.tail = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(ch, 4, 3, padding=1),
            nn.PixelShuffle(2),
        )

    def forward(self, x):
        x_bic = F.interpolate(x, scale_factor=2, mode='bicubic', align_corners=False)

        e1 = self.enc1(self.head(x))
        e2 = self.enc2(self.down1(e1))
        m  = self.mid(self.down2(e2))

        d2 = self.fuse2(torch.cat([self.up2(m), e2], 1))
        d2 = self.dec2(d2)

        d1 = self.fuse1(torch.cat([self.up1(d2), e1], 1))
        d1 = self.dec1(d1)

        return self.tail(d1) + x_bic

In [4]:

model = AdvancedUNetSR(ch=32, n_blocks=4).to(DEVICE)

state_dict = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {n_params:,} parameters")

Model loaded: 3,093,144 parameters


In [5]:

@torch.no_grad()
def predict_with_tta(model, lr_np):
    """
    Input:  lr_np — numpy array (128, 128) float32
    Output: pred  — numpy array (256, 256) float32
    """
    lr_t = torch.from_numpy(lr_np).float().unsqueeze(0).unsqueeze(0)  # (1,1,128,128)

    if not USE_TTA:
        pred = model(lr_t.to(DEVICE)).cpu()
        return pred.squeeze().numpy()

    # 8× self-ensemble: 4 rotations × 2 flips
    preds = []
    for do_flip in (False, True):
        for k in range(4):
            x = lr_t.clone()
            if do_flip:
                x = torch.flip(x, [-1])
            x = torch.rot90(x, k, [-2, -1])

            p = model(x.to(DEVICE)).cpu()

            p = torch.rot90(p, -k, [-2, -1])
            if do_flip:
                p = torch.flip(p, [-1])
            preds.append(p)

    return torch.stack(preds).mean(0).squeeze().numpy()

In [6]:

print("\n" + "=" * 60)
print("  RUNNING INFERENCE")
print("=" * 60)

t0 = time.time()
n_total = len(test_files)

for i, fname in enumerate(test_files):
    # Load test image
    lr = np.load(os.path.join(TEST_LR, fname)).astype(np.float32)

    # Predict
    pred = predict_with_tta(model, lr)

    # Post-process
    pred = np.clip(pred, 0.0, 1.0).astype(np.float32)

    # Validate
    assert pred.shape == (256, 256), f"Shape error: {pred.shape}"
    assert pred.dtype == np.float32, f"Dtype error: {pred.dtype}"
    assert not np.isnan(pred).any(), f"NaN in {fname}"
    assert not np.isinf(pred).any(), f"Inf in {fname}"

    # Save
    np.save(os.path.join(SUB_DIR, fname), pred)

    if (i + 1) % 50 == 0 or (i + 1) == n_total:
        elapsed = time.time() - t0
        speed = (i + 1) / elapsed
        eta = (n_total - i - 1) / speed
        print(f"  {i+1:4d}/{n_total}  "
              f"({speed:.1f} img/s, ETA {eta:.0f}s)")

elapsed = time.time() - t0
print(f"\n  Inference complete: {n_total} images in {elapsed:.1f}s")


  RUNNING INFERENCE
    50/200  (10.5 img/s, ETA 14s)
   100/200  (11.7 img/s, ETA 9s)
   150/200  (12.1 img/s, ETA 4s)
   200/200  (12.4 img/s, ETA 0s)

  Inference complete: 200 images in 16.1s


In [7]:

print("\n" + "=" * 60)
print("  CREATING SUBMISSION CSV")
print("=" * 60)

submission_dir = "/kaggle/working/submission"

rows = []
files = sorted([f for f in os.listdir(submission_dir) if f.endswith(".npy")])

for idx, file in enumerate(files, start=1):
    path = os.path.join(submission_dir, file)

    # Load numpy array
    arr = np.load(path)

    # Convert array to bytes
    buffer = BytesIO()
    np.save(buffer, arr)
    encoded = base64.b64encode(buffer.getvalue()).decode()

    rows.append({
        "id": idx,
        "npy_base64": encoded
    })

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/submission.csv", index=False)

print(f"  Submission created with {len(df)} rows")
print(f"  CSV size: {os.path.getsize('/kaggle/working/submission.csv') / 1e6:.1f} MB")
print(f"  Preview:")
print(df.head())


  CREATING SUBMISSION CSV
  Submission created with 200 rows
  CSV size: 69.9 MB
  Preview:
   id                                         npy_base64
0   1  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
1   2  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
2   3  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
3   4  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...
4   5  k05VTVBZAQB2AHsnZGVzY3InOiAnPGY0JywgJ2ZvcnRyYW...


In [8]:

print("\n" + "=" * 60)
print("  VERIFICATION")
print("=" * 60)

# Check all test images are present
test_count = len([f for f in os.listdir(TEST_LR) if f.endswith(".npy")])
sub_count  = len([f for f in os.listdir(SUB_DIR) if f.endswith(".npy")])
csv_count  = len(df)

print(f"  Test images:       {test_count}")
print(f"  Prediction files:  {sub_count}")
print(f"  CSV rows:          {csv_count}")

assert sub_count == test_count, f"Missing predictions! {sub_count} vs {test_count}"
assert csv_count == test_count, f"CSV row count mismatch! {csv_count} vs {test_count}"

# Spot-check a random prediction
random_file = random.choice(files)
arr = np.load(os.path.join(SUB_DIR, random_file))
print(f"\n  Spot check ({random_file}):")
print(f"    Shape: {arr.shape}")
print(f"    Dtype: {arr.dtype}")
print(f"    Range: [{arr.min():.4f}, {arr.max():.4f}]")
print(f"    Mean:  {arr.mean():.4f}")

print("\n  ✓ ALL CHECKS PASSED — READY TO SUBMIT")
print("=" * 60)


  VERIFICATION
  Test images:       200
  Prediction files:  200
  CSV rows:          200

  Spot check (000163.npy):
    Shape: (256, 256)
    Dtype: float32
    Range: [0.0000, 1.0000]
    Mean:  0.5280

  ✓ ALL CHECKS PASSED — READY TO SUBMIT
